# Dot Plot

Der Dot Plot ist eine einfache Möglichkeit Daten zu visualisieren. Dabei werden einzelne Datenpunkte als Punkte auf einer Achse platziert. Dot Plots eignen sich besonders gut zur Darstellung von Trends, Gruppierungen oder Verteilungen in einem Datensatz. 
Dot Plots sind ähnlich zu Histogrammen, da sie ebenfalls die Verteilung von Daten zeigen. Im Gegensatz zu Histogrammen, die Balken verwenden, behalten Dot Plots jedoch die individuellen Datenpunkte sichtbar bei.

Es gibt verschiedene Varianten von Dot Plots, darunter:

* <b>Cleveland Dot Plots:</b> Zeigen Werte für verschiedene Kategorien mit einzelnen Punkten.
* <b>Wilkinson Dot Plots:</b> Verteilen Punkte gleichmäßig, um Häufigkeiten anzuzeigen.
* <b>Ranged Dot Plots:</b> Verbinden zwei Punkte innerhalb einer Kategorie, um Veränderungen zu visualisieren (z. B. zwischen zwei Jahren).
* <b>Stacked Dot Plots:</b> Stapelt Punkte vertikal oder horizontal übereinander, um die Häufigkeit von Werten darzustellen.


## 1. Für welche Daten eignet sich die Visualisierung?

Ein Dot Plot ist eine Visualisierung, die besonders gut geeignet ist für:
* <i>Zeitliche Entwicklungen</i> (z. B. CO₂-Intensität pro Tag)
* <i>Vergleich von Verteilungen</i> (z. B. Preise verschiedener Produkte)
* <i>Kategorische Daten mit vielen Einzelwerten</i>
* <i>Daten mit hoher Dichte, da Dot Plots Überlappungen reduzieren</i>

.


## 2. Ein Dot Plot besteht aus:

* ~~<b>X-Achse:</b> Eine numerische oder kategoriale Variable (z.B. Monaten).~~
* ~~<b>Y-Achse:</b> Eine numerische Variable (z.B. CO₂-Intensität).~~
* <b>Punkte:</b> Jeder Punkt steht für einen Datenwert und kann in Farbe oder Position variieren. Die Punkte können außerdem horizontal oder vertikal gestapelt werden.

Eine der beiden Achsen beinhaltet eine numerische Variable, die andere eine kategorielle. Ein Beispiel ist, die X-Achse ist die kategorielle und beinhaltet Monate und die Y-Achse zeigt numerisch die CO₂-Intensität an. 

Dot Plots sind oft übersichtlicher als Histogramme, da sie einzelne Werte präziser darstellen.


## 3. Besonderheiten eines Dotplots

* Helfen bei der visuellen Detektion von Outliern.
* Dot Plots können mehrere Kategorien einfach und gut erkennbar vergleichen.
* Dot Plots sind eine Alternative zu <b>Balkendiagrammen</b> oder <b>Boxplots</b>, falls eine genauere Darstellung der einzelnen Datenpunkte wichtig ist.
* Dot Plots eignen sich besonders gut für kleinere Datensätze, da eine zu große Anzahl an Datenpunkten schnell unübersichtlich werden kann.




In [3]:
import pandas as pd
import altair as alt

# CSV-Datei laden
file_path = "DE_2024_monthly.csv"  # Passe den Pfad an, falls nötig
df = pd.read_csv(file_path)

# Spalten umbenennen (Leerzeichen entfernen)
df = df.rename(columns={
    "Datetime (UTC)": "Date",
    "Carbon Intensity gCO₂eq/kWh (direct)": "Carbon_Intensity",
    "Renewable Percentage": "Renewable_Percentage"
})

# Sicherstellen, dass das Datum als Datumswert erkannt wird
df["Date"] = pd.to_datetime(df["Date"])

# Dot Plot erstellen
chart = alt.Chart(df).mark_circle(size=60, opacity=0.7).encode(
    alt.X('Date:T', title='Date'),
    alt.Y('Carbon_Intensity:Q', title='CO₂-Intensität (gCO₂/kWh)'),
    alt.Color('Renewable_Percentage:Q', title='Erneuerbare Energie (%)', scale=alt.Scale(scheme='viridis')),
    alt.Tooltip(['Date:T', 'Carbon_Intensity:Q', 'Renewable_Percentage:Q'])
).interactive()

chart.show()

alt.Chart(...)

### Ranged Dot Plot: Unterschied des Anteils erneuerbarer Energie im Strommix vom Jahr 2021 und 2024

In [ ]:
df_2024 = pd.read_csv("DE_2024_monthly.csv")
df_2021 = pd.read_csv("DE_2021_monthly.csv")

# Combine both years and convert dates to Months and Years
combined_df = pd.concat([df_2024, df_2021], ignore_index=True)
dates = pd.to_datetime(combined_df["Datetime (UTC)"])
months = dates.dt.strftime('%B')
years = dates.dt.strftime('%Y')
combined_df['Months']=pd.Series(months, index=combined_df.index)
combined_df['Years']=pd.Series(years, index=combined_df.index)

# Range Plot
month_order = [
    "January", "February", "March", "April", "May", "June", 
    "July", "August", "September", "October", "November", "December"
]

chart = (
    alt.Chart(combined_df)
    .encode(
        x=alt.X("Renewable Percentage:Q").scale(zero=False), 
        y=alt.Y("Months:N", sort=month_order)
        )
    .transform_filter(alt.FieldOneOfPredicate(field="Years", oneOf=[2021, 2024]))
).properties(
    title="Anteil erneuerbarer Energie im Strommix des Jahres 2021 vs. 2024",
    width=600,
    height=400
)
lines = alt.Chart(combined_df).mark_line(strokeWidth=2).encode(
    x="Renewable Percentage:Q",
    y=alt.Y("Months:N", sort=month_order),
    detail="Months:N",
    color=alt.value("gray"),
)
color = alt.Color("Years:O").scale(domain=[2021, 2024], range=["#e6959c", "#911a24"])
points = (
    chart.mark_point(
        size=100,
        opacity=1,
        filled=True,
    )
    .encode(color=color)
)
(lines + points)


alt.LayerChart(...)

### Cleveland Dot Plot: Vergleich der CO₂-Intensität zwischen verschiedenen EU Ländern (nicht alle)

Hier sind im linken Teil des Grapen die Länder gut zu erkennen, die weniger CO₂ intensiv wirtschaften als die Anderen. Länder in der Mitte haben dabei einen mittleren bis hohen Verbrauch an CO₂.
Tschechien als aleiniger Spitzenreiter ist als Ausreiser zu erkennen, da kein anderes Land aus den hier verwendeten einen so hohen CO₂ ausstoß aufweist.

In [27]:
import os
        
folder_path = "DataEurop"
files = os.listdir(folder_path)
df_list = [pd.read_csv("DataEurop/" + file) for file in files]
dfCI = pd.concat(df_list, ignore_index=True)

dfCI = dfCI.rename(columns={
    "Carbon Intensity gCO₂eq/kWh (direct)": "Carbon_Intensity",
})

alt.Chart(dfCI, title="Vergleich der CO₂-Intensität zwischen verschiedenen EU Ländern").mark_point(size=100, filled=True).encode(
    alt.X('Carbon_Intensity:Q',title="Kohlenstoffintensität (gCO₂/kWh)", scale=alt.Scale(zero=False), axis=alt.Axis(grid=False)),
    alt.Y('Country:N', title='Länder', sort='ascending', axis=alt.Axis(grid=True)),
    color=alt.Color('Country:N')
).properties(
    width=600,
    height=400
).configure_view(stroke="transparent")




alt.Chart(...)

In [2]:
import pandas as pd
import altair as alt

# Dateien laden
files = ["DE_2021_monthly.csv", "DE_2022_monthly.csv", "DE_2023_monthly.csv", "DE_2024_monthly.csv"]
df_list = [pd.read_csv(file) for file in files]

# Alle Jahre zusammenfügen
df = pd.concat(df_list, ignore_index=True)

# Spalten umbenennen (Leerzeichen entfernen)
df = df.rename(columns={
    "Datetime (UTC)": "Date",
    "Carbon Intensity gCO₂eq/kWh (direct)": "Carbon_Intensity",
    "Renewable Percentage": "Renewable_Percentage"
})

# Sicherstellen, dass das Datum als Datumswert erkannt wird
df["Date"] = pd.to_datetime(df["Date"])
df["Year"] = df["Date"].dt.year.astype(str)  # Jahr als String für die Auswahl
df["Month"] = df["Date"].dt.strftime("%b")  # Monatsnamen (Jan, Feb, Mär ...)

# Interaktive Auswahl für das Jahr
selector = alt.param(
    name="year_selector",
    bind=alt.binding_radio(options=["2021", "2022", "2023", "2024"], name="Jahr: "),
    value="2024"  # Standardmäßig 2024 hervorheben
)

# Dot Plot erstellen
chart = alt.Chart(df).mark_circle(size=60).encode(
    x=alt.X("Month:N", title="Monat", sort=["Jan", "Feb", "Mär", "Apr", "Mai", "Jun", "Jul", "Aug", "Sep", "Okt", "Nov", "Dez"]),
    y=alt.Y("Carbon_Intensity:Q", title="CO₂-Intensität (gCO₂/kWh)"),
    color=alt.Color("Renewable_Percentage:Q", title="Erneuerbare Energie (%)", scale=alt.Scale(scheme="viridis")),
    opacity=alt.condition(
        f"datum.Year == year_selector", alt.value(1.0), alt.value(0.3)  # Hervorhebung des gewählten Jahres
    ),
    tooltip=["Date:T", "Carbon_Intensity:Q", "Renewable_Percentage:Q", "Year"]
).properties(
    title="Monatliche Kohlenstoffintensität – Alle Jahre mit Hervorhebung",
    width=800,
    height=400
).add_params(selector)

chart.show()


FileNotFoundError: [Errno 2] No such file or directory: 'DE_2022_monthly.csv'

In [ ]:
import pandas as pd
import altair as alt

# Datei einlesen
df = pd.read_csv("DE_2024_daily.csv")

# Datumsspalte umwandeln
df["Datetime (UTC)"] = pd.to_datetime(df["Datetime (UTC)"])

# Dot Plot erstellen
chart = alt.Chart(df).mark_circle().encode(
    x=alt.X("Datetime (UTC):T", title="Datum"),
    y=alt.Y("Carbon Intensity gCO₂eq/kWh (direct):Q", title="Kohlenstoffintensität (gCO₂/kWh)"),
    tooltip=["Datetime (UTC)", "Carbon Intensity gCO₂eq/kWh (direct)"]
).properties(
    title="Dot Plot der Kohlenstoffintensität über die Zeit",
    width=800,
    height=400
)

chart

alt.Chart(...)

In [ ]:
import pandas as pd
import altair as alt

# Dateien einlesen
files = ["DE_2021_daily.csv", "DE_2022_daily.csv", "DE_2023_daily.csv", "DE_2024_daily.csv"]
df_list = [pd.read_csv(file) for file in files]

# Alle Jahre zusammenfügen
df = pd.concat(df_list, ignore_index=True)

# Datumsspalte umwandeln
df["Datetime (UTC)"] = pd.to_datetime(df["Datetime (UTC)"])

# Dot Plot erstellen
chart = alt.Chart(df).mark_circle().encode(
    x=alt.X("Datetime (UTC):T", title="Datum"),
    y=alt.Y("Carbon Intensity gCO₂eq/kWh (direct):Q", title="Kohlenstoffintensität (gCO₂/kWh)"),
    color=alt.Color("year(Datetime (UTC)):N", title="Jahr"),  # Farbige Punkte pro Jahr
    tooltip=["Datetime (UTC)", "Carbon Intensity gCO₂eq/kWh (direct)"]
).properties(
    title="Dot Plot der Kohlenstoffintensität über die Jahre",
    width=800,
    height=400
)

chart


alt.Chart(...)

In [ ]:
import pandas as pd
import altair as alt

# Dateien einlesen
files = ["DE_2021_daily.csv", "DE_2022_daily.csv", "DE_2023_daily.csv", "DE_2024_daily.csv"]
df_list = [pd.read_csv(file) for file in files]

# Alle Jahre zusammenfügen
df = pd.concat(df_list, ignore_index=True)

# Datumsspalte umwandeln
df["Datetime (UTC)"] = pd.to_datetime(df["Datetime (UTC)"])
df["Year"] = df["Datetime (UTC)"].dt.year.astype(str)

# Interaktive Auswahl für das Jahr (mit selection_single)
selector = alt.param(  
    name="year_selector",
    bind=alt.binding_radio(options=["2021", "2022", "2023", "2024"], name="Jahr: "),
    value="2022"  # Standardmäßig 2022 anzeigen
)

# Dot Plot erstellen
chart = alt.Chart(df).mark_circle().encode(
    x=alt.X("Datetime (UTC):T", title="Datum"),
    y=alt.Y("Carbon Intensity gCO₂eq/kWh (direct):Q", title="Kohlenstoffintensität (gCO₂/kWh)"),
    color=alt.condition(f"year_selector == datum.Year", alt.value("blue"), alt.value("lightgray")),
    tooltip=["Datetime (UTC)", "Carbon Intensity gCO₂eq/kWh (direct)"]
).properties(
    title="Interaktiver Dot Plot der Kohlenstoffintensität",
    width=800,
    height=400
).add_params(selector)

chart


alt.Chart(...)

In [ ]:
import pandas as pd
import altair as alt

# Dateien einlesen
files = ["DE_2021_daily.csv", "DE_2022_daily.csv", "DE_2023_daily.csv", "DE_2024_daily.csv"]
df_list = [pd.read_csv(file) for file in files]

# Alle Jahre zusammenfügen
df = pd.concat(df_list, ignore_index=True)

# Datumsspalte umwandeln
df["Datetime (UTC)"] = pd.to_datetime(df["Datetime (UTC)"])
df["Year"] = df["Datetime (UTC)"].dt.year.astype(str)
df["Month"] = df["Datetime (UTC)"].dt.strftime("%b")  # Kürzel für Monate (Jan, Feb, Mär, ...)

# Sortierung der Monate für die X-Achse
month_order = ["Jan", "Feb", "Mär", "Apr", "Mai", "Jun", "Jul", "Aug", "Sep", "Okt", "Nov", "Dez"]

# Interaktive Auswahl für das Jahr
selector = alt.param(  
    name="year_selector",
    bind=alt.binding_radio(options=["2021", "2022", "2023", "2024"], name="Jahr: "),
    value="2022"
)

# Dot Plot erstellen (mit begrenzter x-Achse)
chart = alt.Chart(df).mark_circle().encode(
    x=alt.X("Month:N", title="Monat", sort=month_order),  # Monate als Kategorien
    y=alt.Y("Carbon Intensity gCO₂eq/kWh (direct):Q", title="Kohlenstoffintensität (gCO₂/kWh)"),
    color=alt.condition(f"year_selector == datum.Year", alt.value("blue"), alt.value("lightgray")),
    tooltip=["Datetime (UTC)", "Carbon Intensity gCO₂eq/kWh (direct)", "Year"]
).properties(
    title="Interaktiver Dot Plot der Kohlenstoffintensität",
    width=800,
    height=400
).add_params(selector)

chart


alt.Chart(...)

In [ ]:
import pandas as pd
import altair as alt

# Dateien einlesen
files = ["DE_2021_daily.csv", "DE_2022_daily.csv", "DE_2023_daily.csv", "DE_2024_daily.csv"]
df_list = [pd.read_csv(file) for file in files]

# Alle Jahre zusammenfügen
df = pd.concat(df_list, ignore_index=True)

# Datumsspalte umwandeln
df["Datetime (UTC)"] = pd.to_datetime(df["Datetime (UTC)"])
df["Year"] = df["Datetime (UTC)"].dt.year.astype(str)
df["DayOfYear"] = df["Datetime (UTC)"].dt.dayofyear  # Tag im Jahr (1-365)

# Dot Plot mit gemeinsamer Tagesachse für alle Jahre
chart = alt.Chart(df).mark_circle().encode(
    x=alt.X("DayOfYear:Q", title="Tag des Jahres (1-365)"),
    y=alt.Y("Carbon Intensity gCO₂eq/kWh (direct):Q", title="Kohlenstoffintensität (gCO₂/kWh)"),
    color=alt.Color("Year:N", title="Jahr"),  # Jedes Jahr bekommt eine eigene Farbe
    tooltip=["Datetime (UTC)", "Carbon Intensity gCO₂eq/kWh (direct)", "Year"]
).properties(
    title="Tägliche Kohlenstoffintensität – Alle Jahre auf einer 365-Tage-Skala",
    width=800,
    height=400
)

chart


alt.Chart(...)

In [ ]:
import pandas as pd
import altair as alt

# Dateien einlesen
files = ["DE_2021_daily.csv", "DE_2022_daily.csv", "DE_2023_daily.csv", "DE_2024_daily.csv"]
df_list = [pd.read_csv(file) for file in files]

# Alle Jahre zusammenfügen
df = pd.concat(df_list, ignore_index=True)

# Datumsspalte umwandeln
df["Datetime (UTC)"] = pd.to_datetime(df["Datetime (UTC)"])
df["Year"] = df["Datetime (UTC)"].dt.year.astype(str)
df["DayOfYear"] = df["Datetime (UTC)"].dt.dayofyear  # Tag im Jahr (1-365)

# Interaktive Auswahl für das Jahr
selector = alt.param(  
    name="year_selector",
    bind=alt.binding_radio(options=["2021", "2022", "2023", "2024"], name="Jahr: "),
    value="2022"  # Standardmäßig 2022 hervorheben
)

# Dot Plot mit Hervorhebung des ausgewählten Jahres
chart = alt.Chart(df).mark_circle().encode(
    x=alt.X("DayOfYear:Q", title="Tag des Jahres (1-365)"),
    y=alt.Y("Carbon Intensity gCO₂eq/kWh (direct):Q", title="Kohlenstoffintensität (gCO₂/kWh)"),
    color=alt.condition(
        f"datum.Year == year_selector", alt.value("blue"), alt.value("lightgray")  # Blau für das ausgewählte Jahr, Grau für andere
    ),
    opacity=alt.condition(
        f"datum.Year == year_selector", alt.value(1.0), alt.value(0.3)  # Hervorgehobenes Jahr volle Deckkraft, andere halbtransparent
    ),
    tooltip=["Datetime (UTC)", "Carbon Intensity gCO₂eq/kWh (direct)", "Year"]
).properties(
    title="Tägliche Kohlenstoffintensität – Alle Jahre mit Hervorhebung",
    width=800,
    height=400
).add_params(selector)

chart


alt.Chart(...)

## Fazit

Dot Plots sind eine leistungsfähige Methode, um detaillierte Einzelwerte zu zeigen. Sie eignen sich besonders für zeitliche Daten, kategorische Vergleiche und dichte Punktwolken. Durch Farb- und Größenkodierung lassen sich zusätzliche Informationen integrieren.